In [82]:
download.file("https://raw.githubusercontent.com/lydaosorio-Univalle/codebook/main/dengue_sivigila.csv", "dengue_sivigila.csv")

In [ ]:
library(dplyr)
library(tidyr)
install.packages("labelled")
library(labelled)
library(tidyverse)
install.packages("writexl")
library(writexl)
install.packages("lubridate")
library(lubridate)

**Actividad de Conversión de la ficha epidemiológica de dengue a un diccionario común.**

### Pasos:
1. Importar la base de datos e identificar las variables
2. Asegurar el formato correcto de cada variable
3. Asignar etiquetas de variable (label variable)
4. Asignar etiquetas de valores (label values)
5. Explorar el codebook
6. Construir y exportar tabla del codebook

In [ ]:
# Paso 1. Importar la base de datos e identificar las variables
dengue <- read.csv("dengue_sivigila.csv")
glimpse(dengue)

### 🤔 Mira el resultado que acabas de generar.

1. ¿Qué quiere decir Rows: 69,910?
2. ¿Qué quiere decir Columns: 7?
3. ¿Qué informacion corresponde a cada fila que empieza con $?

**Escribe tu respuesta aquí:** _(doble clic para editar)_

In [85]:
# Paso 2. Asegurar el formato correcto de cada variable

dengue <- dengue |>
  mutate(

    cod_eve   = as.numeric(cod_eve),
    sexo_     = as.character(sexo_),
    edad_     = as.numeric(edad_),
    uni_med_  = as.numeric(uni_med_),
    tip_cas_  = as.numeric(tip_cas_),
    clasfinal = as.numeric(clasfinal),

  )

glimpse(dengue)

Warning message:
“There were 2 warnings in `mutate()`.
The first warning was:
ℹ In argument: `uni_med_ = as.numeric(uni_med_)`.
Caused by warning:
! NAs introduced by coercion
ℹ Run `dplyr::last_dplyr_warnings()` to see the 1 remaining warning.”


Rows: 69,910
Columns: 7
$ cod_eve   <dbl> 210, 210, 210, 210, 210, 210, 210, 210, 210, 210, 210, 210, …
$ edad_     <dbl> 21, 39, 19, 34, 41, 18, 26, 22, 18, 19, 22, 23, 23, 20, 22, …
$ uni_med_  <dbl> 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, …
$ sexo_     <chr> "F", "M", "M", "M", "M", "M", "M", "F", "M", "M", "M", "F", …
$ ini_sin_  <chr> "09 Jan 10", "08 Jan 10", "07 Jan 10", "04 Jan 10", "03 Jan …
$ tip_cas_  <dbl> 2, 3, 3, 2, 2, 3, 2, 3, 2, 3, 2, 2, 2, 2, 3, 3, 2, 2, 3, 2, …
$ clasfinal <dbl> NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, …


### ✋ Antes de investigar
R dice que introdujo NA al convertir `uni_med_` a número.
¿Qué tipo de valores crees que pudieron causar eso? Anota tu hipótesis.
**Escribe tu respuesta aquí:** _(doble clic para editar)_

#### Ahora revisemos NA en la variable clasfinal. El formato de la variable es numérica pero tiene letras NA ¿crees que debemos cambiar NA por un código numérico? ¿Cuál?


In [ ]:
dengue <- dengue |>
  mutate(clasfinal = replace_na(clasfinal, 99))

glimpse(dengue$clasfinal)

 num [1:69910] 99 99 99 99 99 99 99 99 99 99 ...


### Ahora vamos a definir el formato de fecha
### La variable ini_sin tiene valores 09 Jan 10 ¿que estructura de fecha crees que es esta? ¿que tipo de formato tiene?

In [ ]:
class(dengue$ini_sin_)

In [ ]:
dengue <- dengue %>%
   mutate(ini_sin_formateada = dmy(ini_sin_))

class(dengue$ini_sin_formateada)
glimpse(dengue$ini_sin_formateada)

📅 R guarda las fechas internamente como un número (los días transcurridos desde el 1 de enero de 1970) y, al mostrarlas, siempre usa AAAA-MM-DD ¿Qué ventaja tiene que sea una fecha real y no texto?

In [ ]:
# Paso 3. Asignar etiquetas de variable (label variable)
# Las etiquetas describen QUÉ es cada variable. Tomado del texto del instructivo de la ficha de sivigila.

dengue <- dengue |>
  set_variable_labels(
    cod_eve            = "Código del evento (ítem 1.2)",
    sexo_              = "Sexo biológico del paciente (ítem 2.9)",
    edad_              = "Edad cumplida del paciente (ítem 2.6)",
    uni_med_           = "Unidad de medida de la edad (ítem 2.7)",
    ini_sin_           = "Fecha de inicio de síntomas (ítem 3.5)",
    ini_sin_formateada = "Fecha de inicio de síntomas d-m-a",
    tip_cas_           = "Clasificación inicial del caso (ítem 3.6)",
    clasfinal          = "Clasificación final del dengue (ítem 7.1)"
  )

var_label(dengue)  # etiquetas de todas las variables


In [ ]:
# Paso 4. Asignar etiquetas de valores (label values)
# Las etiquetas de valores describen QUÉ SIGNIFICA cada código. Solo aplica a las variables categóricas.
# Sintaxis: c("Etiqueta" = código)

dengue <- dengue |>
  set_value_labels(
    cod_eve   = c("Dengue"                = 210,
                  "Dengue grave"          = 220,
                  "Mortalidad por dengue" = 580),
    uni_med_  = c("Anhos"     = 1,
                  "Meses"     = 2,
                  "Días"      = 3,
                  "Horas"     = 4,
                  "Minutos"   = 5,
                  "No aplica" = 0),
    sexo_     = c("Hombre"        = "M",
                  "Mujer"         = "F",
                  "Indeterminado" = "I"),
    tip_cas_  = c("Sospechoso"                    = 1,
                  "Probable"                      = 2,
                  "Conf.por laboratorio"          = 3,
                  "Conf. por clínica"             = 4,
                  "Conf. por nexo epidemiológico" = 5),
    clasfinal = c("No aplica"                   = 0,
                  "Dengue sin signos de alarma" = 1,
                  "Dengue con signos de alarma" = 2,
                  "Dengue grave"                = 3)
  )

val_labels(dengue)


In [90]:
# 5. Explorar el codebook
# Vista general de variables y etiquetas
print(look_for(dengue))

 pos variable           label              col_type missing values            
 1   cod_eve            Código del evento~ dbl+lbl  0       [210] Dengue      
                                                            [220] Dengue grave
                                                            [580] Mortalidad ~
 2   edad_              Edad cumplida del~ dbl      8                         
 3   uni_med_           Unidad de medida ~ dbl+lbl  14      [1] Anhos         
                                                            [2] Meses         
                                                            [3] Días          
                                                            [4] Horas         
                                                            [5] Minutos       
                                                            [0] No aplica     
 4   sexo_              Sexo biológico de~ chr+lbl  0       [M] Hombre        
                                                    

In [ ]:
#   6. Construir y exportar tabla del codebook
extraer_val_labels <- function(x) {
  labs <- val_labels(x)
  if (is.null(labs) || length(labs) == 0) {
    return("Sin valores etiquetados")
  }
  paste(paste0(labs, " = ", names(labs)), collapse = "; ")
}

codebook_dengue <- tibble(
  columna             = names(dengue),
  tipo                = map_chr(dengue, ~ class(.x)[1]),
  etiqueta            = map_chr(dengue, ~ var_label(.x) %||% "Sin etiqueta"),
  valores_etiquetados = map_chr(dengue, extraer_val_labels)
)

print(codebook_dengue)

write_xlsx(codebook_dengue, "diccionario de datos_dengue.xlsx")